[문제] LogicGate클래스에서 은닉층의 노드 갯수를 4개로 변경하여 XOR 게이트의 결과값이 정상적으로 나오도록 코드를 수정하여 제출하시오.

재출파일명 : 홍길동_0518.ipynb

<처리조건>
1. LogicGate클래스를 설계한다.
2. 학습후 정확도를 출력하도록 한다.

In [17]:
import numpy as np

def sigmoid(x):   # 시그모이드 함수
    return 1. / (1 + np.exp(-x))   # 수식을 표현한 내용

In [18]:
def numerical_derivative(f, x):   # f: 손실값을 계산하는 함수 / x: 업데이트 하기 위한 변수
    delta_x = 1e-4   # 미분할 값의 크기
    grad = np.zeros_like(x)
    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])
    
    while not it.finished:
        idx = it.multi_index        
        tmp_val = x[idx]
        x[idx] = float(tmp_val) + delta_x
        fx1 = f(x) # f(x+delta_x)
        
        x[idx] = float(tmp_val) - delta_x 
        fx2 = f(x) # f(x-delta_x)
        grad[idx] = (fx1 - fx2) / (2*delta_x)
        
        x[idx] = tmp_val 
        it.iternext()

    return grad

In [201]:
class LogicGate:
    def __init__(self, gate_name, xdata, tdata): # 생성자 : __init__ # 해당 클래스가 사용할 변수를 초기화할 목적으로 사용
        self.name = gate_name
        self.xdata = xdata.reshape(4,2)  # 입력층 데이터 // 입력층에 대한 설계
        self.tdata = tdata.reshape(4,1)  # 정답 데이터

        self.W2 = np.random.rand(2,4)    # 은닉층 가중치 // 은닉층에 대한 설계
        self.b2 = np.random.rand(4)      # 은닉층 바이어스

        self.W3 = np.random.rand(4,1)    # 출력증 가중치 // 출력층에 대한 설계
        self.b3 = np.random.rand(1)      # 출력층 바이어스

        self.learning_rate = 1e-2        # 학습률

    def loss_val(self):                  # 손실값 계산 
        delta = 1e-7
        z2 = np.dot(self.xdata, self.W2) + self.b2
        a2 = sigmoid(z2)
        z3 = np.dot(a2, self.W3) + self.b3
        y = a3 = sigmoid(z3)
        return -np.sum(self.tdata*np.log(y+delta)+(1-self.tdata)*np.log((1 - y)+delta)) # 최종적인 y값을 가지고 손실값 계산

# 손실함수
    def train(self):
        f = lambda x : self.loss_val() # 손실함수 정의하는 형태 
        print("Initial loss value = ", self.loss_val()) # 초기손실값 계산하는 형태
        for step in range(13001): # 2만번 학습! >> 학습을 하게 되면 각 층에 있는 가중치와 바이어스를 업데이트 시켜야한다.
            self.W2 -= self.learning_rate * numerical_derivative(f, self.W2) # 은닉층의 가중치와 바이어스를 업데이트 시킴
            self.b2 -= self.learning_rate * numerical_derivative(f, self.b2) # 은닉층의 가중치와 바이어스를 업데이트 시킴
            self.W3 -= self.learning_rate * numerical_derivative(f, self.W3) # 출력층의 가중치와 바이어스를 업데이트 시킴
            self.b3 -= self.learning_rate * numerical_derivative(f, self.b3) # 출력층의 가중치와 바이어스를 업데이트 시킴
            if (step % 1000 == 0):                                           # 중간 확인 과정
                print("step = ", step, "loss value = ", self.loss_val())

    def predict(self, input_data):
        self.xdata = input_data
        z2 = np.dot(self.xdata, self.W2) + self.b2 # 은닉층에서 연산
        a2 = sigmoid(z2)                           # 은닉층에서 연산
        z3 = np.dot(a2, self.W3) + self.b3         # 출력층에서 연산
        y = a3 = sigmoid(z3)                       # 출력층에서 연산
        if y > 0.5:
            result = 1
        else:
            result = 0

        return y, result

    # 정확도 계산하는 거 만들 거임
    def accuracy(self, test_xdata, test_tdata):
        matched_list = []                                               # 정답과 일치한 내용 저장
        not_matched_list = []                                           # 정답과 불일치한 내용 저장
        for index in range(len(test_xdata)):
            (real_val, logical_val) = self.predict(test_xdata[index])
            if logical_val == test_tdata[index]:
                matched_list.append(index)                              # 정답과 일치한 내용 추가
            else:
                not_matched_list.append(index)                          # 정답과 불일치한 내용 추가
        accuracy_val = len(matched_list) / len(test_xdata)              # 정확도 계산
        return accuracy_val

In [206]:
# XOR 게이트
# XOR
xdata = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])   # 학습(train) 데이터
tdata = np.array([0, 1, 1, 0])                         # 정답(target) 데이터
XOR_obj = LogicGate("XOR_GATE", xdata, tdata)          # 모델 객체 생성
XOR_obj.train()                                        # 학습

Initial loss value =  4.567253066975287
step =  0 loss value =  4.487906497146844
step =  1000 loss value =  2.750333625311284
step =  2000 loss value =  2.732058503906234
step =  3000 loss value =  2.694650760679271
step =  4000 loss value =  2.612118989770262
step =  5000 loss value =  2.4483722047964123
step =  6000 loss value =  2.2211724943304985
step =  7000 loss value =  1.9700353587755388
step =  8000 loss value =  1.6254045323295536
step =  9000 loss value =  1.1434768249749503
step =  10000 loss value =  0.7314435769983821
step =  11000 loss value =  0.4876396266536166
step =  12000 loss value =  0.35090153238050276
step =  13000 loss value =  0.26892423646513447


In [207]:
test_data = np.array([ [0, 0], [0, 1], [1, 0], [1, 1] ])
for data in test_data:
    sigmoid_val, logical_val = XOR_obj.predict(data)
    print(data, '=', logical_val)

print('----------------------------------------')
test_tdata = np.array([0, 1, 1, 0]) # 정답
accuracy_ret = XOR_obj.accuracy(test_data, test_tdata)
print('Accuracy =>', accuracy_ret)

[0 0] = 0
[0 1] = 1
[1 0] = 1
[1 1] = 0
----------------------------------------
Accuracy => 1.0
